# 01 — Inspect real data and provisional color labels

Run cells in order using the project `.venv`. This notebook reads four samples;
it does not load diffusion weights or train. These small previews can run on CPU.
Change settings in `configs/pilot.json`, or override `config` below.

We split **image IDs** before selecting hair/clothing, so regions from the same
photo stay together. Labels and the initial color palette need visual review,
especially on patterned clothing. This is a data check, not an evaluation score.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/pilot.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the project repository, then reopen this notebook.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = str(ROOT / "results/_hf_cache")
config = json.loads((ROOT / "configs/pilot.json").read_text())
SHARED = Path(config["shared_root"])
DATA = SHARED / "data/CelebAMask-HQ"
MODEL = SHARED / "models/sd15-inpainting"
if not DATA.is_dir() or not MODEL.is_dir():
    raise FileNotFoundError("Check shared_root in configs/pilot.json.")
from IPython.display import display
from PIL import Image
from color_editing.data import select_samples, describe_sample, contact_sheet
from color_editing.runs import create_run, save_json, finish_run, publish_run
print(json.dumps(config, indent=2))
# To try clothing, set config["region"] = "cloth" here before selecting samples.

## Select four training examples

Only masks large enough for `min_mask_fraction` are selected. Missing class masks
are skipped. `mask_dilation=5` means a 5 × 5 maximum-filter kernel, not a radius.

In [ ]:
from color_editing.data import COLOR_RGB, split_ids

samples = select_samples(
    DATA, region=config["region"], split="train", limit=config["preview_samples"],
    seed=config["split_seed"], min_mask_fraction=config["min_mask_fraction"],
    dilation=config["mask_dilation"],
)
records = [describe_sample(s) for s in samples]
print("Split sizes:", {key: len(ids) for key, ids in split_ids(config["split_seed"]).items()})
print("Selected image IDs:", [s["image_id"] for s in samples])

## Inspect alignment, selected region, and color guess

For each example: original, raw mask, dilated mask, isolated region, and named-color
prototype. Photos are resized to 512 × 512. White pixels select the region.
The nearest median-Lab label is a heuristic, so inspect the region and swatch.

In [ ]:
previews = []
for sample, record in zip(samples, records):
    region_view = Image.composite(
        sample["image"], Image.new("RGB", (512, 512), "white"), sample["mask"],
    )
    swatch = Image.new("RGB", (512, 512), COLOR_RGB[record["source_color"]])
    board = contact_sheet(
        [sample["image"], sample["mask"], sample["edit_mask"], region_view, swatch],
        [f"ID {sample['image_id']} / {sample['region']}", "original mask", "dilated mask",
         "selected region", f"provisional: {record['source_color']}"],
    )
    previews.append(board)
    display(board)
    print(json.dumps(record, indent=2))

## Save this inspection

This cell creates a new personal `results/` run containing the exact split,
selected sample metadata, initial palette, previews, config, and provenance.
It leaves shared originals unchanged. Re-running creates another run.

In [ ]:
RUN = create_run("data-preview", config)
try:
    save_json(RUN / "splits.json", split_ids(config["split_seed"]))
    save_json(RUN / "samples.json", records)
    save_json(RUN / "palette.json", COLOR_RGB)
    (RUN / "previews").mkdir()
    for sample, board in zip(samples, previews):
        board.save(RUN / "previews" / f"{sample['image_id']:05d}-{sample['region']}.png")
    finish_run(RUN, samples=len(samples), label_status="provisional_visual_review")
except BaseException as error:
    finish_run(RUN, status="failed", error=str(error))
    raise
print("Saved inspection:", RUN)

## Optional team copy

Code goes through GitHub. Selected completed runs can be copied within Ocean to
`shared/runs/<username>/<run-id>/`; this cell is commented out by default.

In [ ]:
# Optional: uncomment only when this completed run is ready for the team.
# shared_run = publish_run(RUN, SHARED)
# print(shared_run)